# Step 1: Environment Setup and Pre-trained Model Loading

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
project_dir = "/content/drive/MyDrive/Brain_Tumor_Classification"
%cd {project_dir}

# Ensure code runs on CPU as per runtime adjustment
device = torch.device('cpu')
print(f"Training Environment set to: {device}")

# Load pre-trained ResNet18 weights
weights = models.ResNet18_Weights.DEFAULT
model_resnet = models.resnet18(weights=weights)

print("Pre-trained ResNet18 model loaded successfully.")

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1Rd5_epoDI2porCXmS9-4opjNV4Tdx9s2/Brain_Tumor_Classification
Training Environment set to: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s]


Pre-trained ResNet18 model loaded successfully.


# Step 2: Architecture Modification (Fine-Tuning for 4 Classes)

In [3]:
# Freeze all pre-trained layers to retain feature extraction capabilities
for param in model_resnet.parameters():
    param.requires_grad = False

# Extract the number of input features from the original final layer
num_ftrs = model_resnet.fc.in_features

# Replace the final fully connected layer with a new one tailored for 4 tumor classes
model_resnet.fc = nn.Sequential(
    nn.Linear(num_ftrs, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 4) # 4 output classes: glioma, meningioma, pituitary, no_tumor
)

# Move the modified model to the active device
model_resnet = model_resnet.to(device)
print(model_resnet.fc)
print("Final layer successfully modified for 4-class classification.")

Sequential(
  (0): Linear(in_features=512, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=128, out_features=4, bias=True)
)
Final layer successfully modified for 4-class classification.


# Step 3: Optimization Configuration and Transfer Learning Verification

In [4]:
# Define Cross-Entropy Loss for multi-class classification
criterion = nn.CrossEntropyLoss()

# Optimize ONLY the parameters of the newly added fully connected layer
optimizer = optim.Adam(model_resnet.fc.parameters(), lr=0.001)

# Verification pipeline to ensure dimensions align perfectly
print("Verifying forward pass with modified ResNet18 architecture...")
try:
    # Generate a dummy tensor representing a batch of 2 images: [Batch_Size, Channels, Height, Width]
    dummy_input = torch.randn(2, 3, 224, 224).to(device)
    dummy_output = model_resnet(dummy_input)
    print(f"Output Tensor Shape: {dummy_output.shape}")
    # Expected output shape: [2, 4] -> 2 images, 4 class probabilities each
    print("Step 4 Transfer Learning pipeline is fully operational and error-free!")
except Exception as e:
    print(f"Verification failed due to error: {e}")

Verifying forward pass with modified ResNet18 architecture...
Output Tensor Shape: torch.Size([2, 4])
Step 4 Transfer Learning pipeline is fully operational and error-free!
